# Stress Response, Modern Era: the fleet as it is now

**Why this exists.** Notebook 05 measures the GB battery fleet against operator-grade scarcity
over 2018–2026. Notebook 07 shows that window contains a structural break: the fleet's response
steps up around the Open Balancing Platform cutover, and the change survives controls for
composition and for system tightness.

That makes notebook 05's headline numbers a **blend of two regimes**, and the blend is weighted
toward the old one — 23 of its 35 quarters sit before the break. A reader asking "how does the
GB battery fleet behave under stress?" is asking about the fleet that exists now, and gets an
answer averaged with a fleet that was routinely skipped in dispatch.

This notebook runs the same measurements on the **modern era only**, from 2024-04-01. Nothing
methodological changes: same store, same conditioning sets, same normalisation. Only the window.

**What that buys and what it costs.** It buys numbers that describe the current fleet without
a five-year tail. It costs statistical power — roughly a third of the periods, and only two
winters — so the sets that were already thin in notebook 05 become case studies here or vanish.
Every table below carries its own n for that reason, and where a set is too small to support a
distribution it is reported as a count rather than a mean.


In [1]:
%matplotlib inline

import datetime as dt
import importlib.util
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from fleet import census
from fleet import performance as fleet_perf
from fleet.population import census_population
from live import resilience
from live.assets import bess_config

_spec = importlib.util.spec_from_file_location(
    "build_stress_store", REPO_ROOT / "scripts" / "build_stress_store.py"
)
bss = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(bss)

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
})
C = {"ink": "#0b0b0b", "cost": "#e34948", "soc": "#4a3aa7", "ghost": "#c3c2b7",
     "discharge": "#1baf7a", "mid": "#c98500"}

# Same frozen vintage as notebooks 04-07.
census.SNAPSHOT = dt.date(2026, 8, 24)

# The modern era, as notebook 07 defines it: after the Open Balancing Platform
# transition months, which that notebook holds out rather than assigning.
MODERN_START = pd.Timestamp("2024-04-01", tz="UTC")
FULL_START, FULL_END = dt.date(2018, 1, 1), dt.date(2026, 8, 24)

HH = 0.5
CFG = bess_config()
ETA_C, ETA_D = CFG["charge_efficiency"], CFG["discharge_efficiency"]
MIN_N_DIST = 10          # below this a set is a case study, not a distribution
DRM_CRITICAL_MW = 1000.0

POP = census_population()
SITE_MW = {s.site: s.power_mw for s in POP.sites}
SITE_MWH = {s.site: s.capacity_mwh for s in POP.sites}
STORE = bss.store_for(POP)

print(f"Modern era : {MODERN_START.date()} → {FULL_END}")
print(f"Population : {len(SITE_MW)} sites, {sum(SITE_MW.values()):,.0f} MW declared")
print(f"Efficiency : ηc={ETA_C:.2f} ηd={ETA_D:.2f}")


Worksheet row for GB-BESS-WOLVB (Wolverhampton West BESS) needs review: 310 MWh over 56.0 MW declared implies 5.5 h, longer than any priced census site. Self-consistent with the duration recorded, so the agreement check cannot judge it — confirm the figure covers this BM Unit and not the wider project.


Modern era : 2024-04-01 → 2026-08-24
Population : 87 sites, 6,234 MW declared
Efficiency : ηc=0.94 ηd=0.94


## 1. The fleet's position, modern era

The store spans the full window; it is trimmed to the modern era **after** the pre-battery
correction, so a reused connection point cannot leak in. Response is normalised by the
nameplate online at each moment, as everywhere else in this project.


In [2]:
S = bss.load_store(STORE)
pn_all = S["fleet_pn"].copy()
mels_all = S["fleet_mels"].copy()
system = S["system"]
prints = S["lolpdrm_prints"]

for frame in (pn_all, mels_all):
    frame["time"] = pd.to_datetime(frame["time"], utc=True)
pn_all = pn_all[pn_all["site"].isin(SITE_MW)]
mels_all = mels_all[mels_all["site"].isin(SITE_MW)]

# Connection points get reused; drop each site's pre-battery history first.
ERA_START = fleet_perf.battery_era_start(pn_all, SITE_MW)
for frame_name, frame in (("pn", pn_all), ("mels", mels_all)):
    keep = pd.Series(True, index=frame.index)
    for site, valid_from in ERA_START.items():
        keep &= ~((frame["site"] == site) & (frame["time"] < valid_from))
    if frame_name == "pn":
        pn_all = frame[keep]
    else:
        mels_all = frame[keep]

pn = pn_all[pn_all["time"] >= MODERN_START].copy()
mels = mels_all[mels_all["time"] >= MODERN_START].copy()
pn["date"] = pn["time"].dt.date

span = pn.groupby("site")["date"].agg(["min", "max"])
days = pd.DatetimeIndex(sorted(pn["date"].unique()), tz="UTC")
online_mw = pd.Series(0.0, index=days)
for site, row in span.iterrows():
    live = (days.date >= row["min"]) & (days.date <= row["max"])
    online_mw[live] += SITE_MW.get(site, 0.0)

fleet_net = pn.groupby("time")["mw"].sum()
grid = fleet_net.index
online_at = pd.Series(online_mw.reindex(grid.normalize()).to_numpy(), index=grid)
norm_net = fleet_net / online_at.replace(0, np.nan)
avail_factor = mels.groupby("time")["mw"].sum().reindex(grid) / online_at.replace(0, np.nan)

print(f"periods        : {len(grid):,}")
print(f"active sites   : {pn['site'].nunique()}")
print(f"online nameplate: {online_mw.iloc[0]:,.0f} MW → {online_mw.iloc[-1]:,.0f} MW")


periods        : 42,046
active sites   : 80
online nameplate: 2,130 MW → 5,972 MW


## 2. Conditioning sets

Identical rules to notebook 05, with the thresholds recomputed **within the modern era**. That
is the point: "the tightest 1% of periods" should mean the tightest 1% of the fleet's own era,
not a threshold inherited from years when the system ran tighter.


In [3]:
flags = resilience.classify_periods(system["residual_mw"])
final = (prints.sort_values(["horizon", "publish_time"], ascending=[True, False])
               .drop_duplicates("time").set_index("time")[["lolp", "drm_mw"]].sort_index())
cmn = S["cmn"]
cmn_issued = cmn[cmn["type_id"] == 1] if not cmn.empty else cmn
tiers = resilience.classify_tiers(flags, final, cmn_issued)

tiers = tiers[tiers.index >= MODERN_START]
tgrid = tiers.index.intersection(grid)
tiers = tiers.reindex(tgrid)
known = tiers["tier2_known"]
lolp, drm = tiers["lolp"], tiers["drm_mw"]

lolp_p99 = float(lolp[known].quantile(0.99)) if known.any() else float("nan")
C_LOLP = ((lolp >= lolp_p99) if lolp_p99 > 0 else (lolp > 0)).fillna(False)
drm_p01 = float(drm[known].quantile(0.01)) if known.any() else float("nan")
C_DRM = (drm <= drm_p01).fillna(False)
C_DRM1000 = (drm < DRM_CRITICAL_MW).fillna(False)
C_CMN = tiers["tier3"].fillna(False)
C_TIER1 = tiers["tier1"].fillna(False)

SETS = {
    "All periods": pd.Series(True, index=tgrid),
    "Tier 1 (residual decile)": C_TIER1,
    "C_LOLP": C_LOLP,
    "C_DRM (p1)": C_DRM,
    "C_DRM<1GW": C_DRM1000,
    "C_CMN": C_CMN,
}
CRITICAL = (C_LOLP | C_DRM | C_DRM1000).fillna(False)

print(f"LoLP p99 (modern) : {lolp_p99:.3e}")
print(f"DRM p1   (modern) : {drm_p01:,.0f} MW    (min {drm.min():,.0f} MW)")
print(f"Critical union    : {int(CRITICAL.sum())} periods\n")
print(pd.Series({k: int(v.sum()) for k, v in SETS.items()}, name="periods").to_string())


LoLP p99 (modern) : 7.855e-06
DRM p1   (modern) : 5,260 MW    (min 239 MW)
Critical union    : 569 periods

All periods                 42046
Tier 1 (residual decile)     2397
C_LOLP                        421
C_DRM (p1)                    421
C_DRM<1GW                       2
C_CMN                           3


## 3. State of charge

Elexon publishes no state of charge, so it is integrated from the notified position. Both the
integration and the usability filters live in `fleet.performance` — notebooks 05 and 08 share
one implementation, because two integrations of the same series would be two answers.

A site qualifies only if it cycles enough to carry information and its integration is not
pinned at a bound half the time. The re-anchored variant (a daily 04:00 reset) is carried as a
sensitivity throughout: it stops error accumulating but imposes a level nobody observed, so the
two bracket the answer rather than one being right.


In [4]:
soc = fleet_perf.fleet_state_of_charge(
    pn, grid, SITE_MWH, SITE_MW, ETA_C, ETA_D, hours_per_period=HH
)
fleet_soc = soc["soc"]
fleet_soc_anchor = soc["soc_anchored"]
USABLE_SOC = soc["usable"]

if soc["skipped_no_mwh"]:
    print(f"No published energy capacity for {len(soc['skipped_no_mwh'])} active sites "
          f"({soc['skipped_mw']:,.0f} MW) — excluded from every SoC figure, kept in all "
          f"MW-normalised ones.\n")
print(f"SoC-usable sites: {len(USABLE_SOC)} of {len(soc['diagnostics'])} with a known MWh")
print(soc["diagnostics"].head(12).round(3).to_string())


No published energy capacity for 17 active sites (591 MW) — excluded from every SoC figure, kept in all MW-normalised ones.

SoC-usable sites: 46 of 63 with a known MWh
                      capacity_mwh  cycles_per_day  clamp_frac  usable_soc  periods
site                                                                               
Roaring Hill BESS           74.985           1.769       0.249        True    42046
Erskine BESS                30.000           1.736       0.235        True     1968
Blackhillock               400.000           1.564       0.326        True    40606
Holes Bay Battery           10.000           1.488       0.170        True    42046
Little Raith BESS           98.000           1.469       0.257        True    42046
North Tawton BESS           30.000           1.444       0.201        True     7630
Pivot Power Coventry        80.000           1.285       0.241        True    42046
Richborough                100.000           1.196       0.262        True 

## 4. Response by conditioning set

The headline table. Same columns as notebook 05, computed on the modern era.


In [5]:
def set_stats(mask, name):
    m = mask.fillna(False)
    n = int(m.sum())
    if n == 0:
        return {"set": name, "n": 0}
    return {
        "set": name,
        "n": n,
        "net_MW_per_MW": float(norm_net.reindex(tgrid)[m].mean()),
        "median_MW_per_MW": float(norm_net.reindex(tgrid)[m].median()),
        "discharging_%": float((fleet_net.reindex(tgrid)[m] > 0).mean()),
        "avail_factor": float(avail_factor.reindex(tgrid)[m].mean()),
        "mean_SoC": float(fleet_soc.reindex(tgrid)[m].mean()) if USABLE_SOC else np.nan,
        "mean_SoC_anchored": (float(fleet_soc_anchor.reindex(tgrid)[m].mean())
                              if USABLE_SOC else np.nan),
    }


rq1 = pd.DataFrame([set_stats(m, k) for k, m in SETS.items()]).set_index("set")
print("Fleet response by conditioning set — modern era")
print(rq1.round(3).to_string())

baseline = rq1.loc["All periods"]
print()
for name in ("C_LOLP", "C_DRM (p1)", "C_DRM<1GW", "C_CMN"):
    row = rq1.loc[name]
    if row["n"] == 0:
        print(f"{name}: no periods in this era")
        continue
    tag = "  [case study — too few for a distribution]" if row["n"] < MIN_N_DIST else ""
    print(f"{name} (n={int(row['n'])}){tag}")
    print(f"    net {row['net_MW_per_MW']:+.3f} MW/MW online vs "
          f"{baseline['net_MW_per_MW']:+.3f} baseline · "
          f"discharging in {row['discharging_%']:.0%} · "
          f"availability {row['avail_factor']:.0%} · SoC {row['mean_SoC']:.0%}")


Fleet response by conditioning set — modern era
                              n  net_MW_per_MW  median_MW_per_MW  discharging_%  avail_factor  mean_SoC  mean_SoC_anchored
set                                                                                                                       
All periods               42046          0.007             0.002          0.516         0.279     0.414              0.429
Tier 1 (residual decile)   2397          0.065             0.044          0.794         0.275     0.427              0.430
C_LOLP                      421          0.115             0.119          0.926         0.266     0.390              0.386
C_DRM (p1)                  421          0.099             0.099          0.907         0.277     0.396              0.401
C_DRM<1GW                     2          0.133             0.133          1.000         0.231     0.485              0.430
C_CMN                         3          0.131             0.173          1.000         0.2

## 5. Events, and how the fleet moves through them

Consecutive critical periods joined into events — gaps of up to an hour bridged, anything
shorter than an hour discarded as a blip. Then the question notebook 05 exists to ask: does the
fleet sustain its response, or empty?


In [6]:
BRIDGE_HH, MIN_EVENT_HH, FLOOR_BAND, DEPLETED = 2, 2, 0.05, 0.23

crit = pd.Series(CRITICAL, index=tgrid).fillna(False)
bridged = crit.rolling(BRIDGE_HH + 1, center=True, min_periods=1).max().astype(bool)
bridged = bridged & crit.rolling(2 * BRIDGE_HH + 1, center=True, min_periods=1).max().astype(bool)
block = (bridged != bridged.shift(fill_value=False)).cumsum()[bridged]

rows, gap_rows = [], []
for _, idx in bridged[bridged].groupby(block):
    times = idx.index
    if len(times) < MIN_EVENT_HH:
        continue
    during = norm_net.reindex(times)
    path = fleet_soc.reindex(times)
    rows.append({
        "hours": len(times) * 0.5,
        "soc_at_onset": path.iloc[0] if len(path) else np.nan,
        "mean_response": during.mean(),
        "first_half": during.iloc[: max(len(during) // 2, 1)].mean(),
        "second_half": during.iloc[len(during) // 2:].mean(),
    })
    if path.notna().any():
        gap_rows.append({"preparedness": max(0.0, 1.0 - path.iloc[0]),
                         "dispatch": path.min(),
                         "duration": (path <= DEPLETED).mean()})

events = pd.DataFrame(rows).dropna(subset=["mean_response"])
gaps3 = pd.DataFrame(gap_rows)
print(f"Stress events (bridged, >= 1h): {len(events)}")
if len(events):
    print(f"  median duration       : {events['hours'].median():.1f} h "
          f"(max {events['hours'].max():.1f} h)")
    print(f"  median SoC at onset   : {events['soc_at_onset'].median():.0%}")
    print(f"  response, first half  : {events['first_half'].mean():+.3f} MW/MW")
    print(f"  response, second half : {events['second_half'].mean():+.3f} MW/MW")
    print(f"  decay across the event: "
          f"{events['second_half'].mean() - events['first_half'].mean():+.3f} MW/MW")
if len(gaps3):
    print(f"\nEvents decomposed: {len(gaps3)}")
    print(f"  preparedness gap : {gaps3['preparedness'].mean():.0%} of usable energy absent at onset")
    print(f"  dispatch gap     : {gaps3['dispatch'].mean():.0%} still held at the deepest point")
    print(f"  duration gap     : {gaps3['duration'].mean():.0%} of event time below {DEPLETED:.0%} SoC")


Stress events (bridged, >= 1h): 97
  median duration       : 3.5 h (max 14.5 h)
  median SoC at onset   : 47%
  response, first half  : +0.119 MW/MW
  response, second half : +0.103 MW/MW
  decay across the event: -0.015 MW/MW

Events decomposed: 97
  preparedness gap : 52% of usable energy absent at onset
  dispatch gap     : 32% still held at the deepest point
  duration gap     : 7% of event time below 23% SoC


## 6. How hard do sites push?

A fleet mean hides the distribution. Under the era's own critical periods, what share of online
sites discharge above a given fraction of their own nameplate?


In [7]:
wide = pn.pivot_table(index="time", columns="site", values="mw", aggfunc="sum")
crit_idx = tgrid[CRITICAL.to_numpy()]
depth_cols = []
for site in wide.columns:
    mw_site = SITE_MW.get(site, 0.0)
    if mw_site <= 0:
        continue
    frac = (wide[site].reindex(crit_idx) / mw_site).dropna()
    if not frac.empty:
        depth_cols.append(frac.rename(site))

depth = pd.concat(depth_cols, axis=1) if depth_cols else pd.DataFrame()
online = depth.notna()
total = max(int(online.sum().sum()), 1)
print(f"Site-periods under critical conditions: {total:,} across {depth.shape[1]} sites")
for lvl in (0.25, 0.50, 0.75):
    print(f"  discharging above {lvl:.0%} of nameplate : "
          f"{(depth >= lvl).sum().sum() / total:.1%} of site-periods")


Site-periods under critical conditions: 36,793 across 80 sites
  discharging above 25% of nameplate : 20.3% of site-periods
  discharging above 50% of nameplate : 11.0% of site-periods
  discharging above 75% of nameplate : 6.0% of site-periods


## 7. Against the full window

Notebook 05's figures are quoted here from its committed outputs, so the two can be read side
by side. They are **not** recomputed — this table is a comparison, and the eight-year numbers
belong to that notebook.


In [8]:
# Notebook 05's published figures, 2018-01-01 → 2026-08-24. Typed once, here,
# purely to sit beside the modern-era results computed above.
FULL_WINDOW = {
    "C_LOLP net (MW/MW)": 0.060,
    "C_LOLP discharging": 0.88,
    "baseline net (MW/MW)": 0.003,
    "events": 421,
    "median event duration (h)": 3.0,
    "median SoC at onset": 0.62,
    "preparedness gap": 0.35,
    "dispatch gap": 0.52,
    "duration gap": 0.09,
    "response first half": 0.057,
    "response second half": 0.051,
    "above 50% of nameplate": 0.110,
}
modern = {
    "C_LOLP net (MW/MW)": rq1.loc["C_LOLP", "net_MW_per_MW"],
    "C_LOLP discharging": rq1.loc["C_LOLP", "discharging_%"],
    "baseline net (MW/MW)": rq1.loc["All periods", "net_MW_per_MW"],
    "events": len(events),
    "median event duration (h)": events["hours"].median(),
    "median SoC at onset": events["soc_at_onset"].median(),
    "preparedness gap": gaps3["preparedness"].mean(),
    "dispatch gap": gaps3["dispatch"].mean(),
    "duration gap": gaps3["duration"].mean(),
    "response first half": events["first_half"].mean(),
    "response second half": events["second_half"].mean(),
    "above 50% of nameplate": (depth >= 0.50).sum().sum() / total,
}
compare = pd.DataFrame({"full window (nb05)": FULL_WINDOW, "modern era (nb08)": modern})
compare["change"] = compare["modern era (nb08)"] - compare["full window (nb05)"]
print(compare.round(3).to_string())


                           full window (nb05)  modern era (nb08)   change
C_LOLP net (MW/MW)                      0.060              0.115    0.055
C_LOLP discharging                      0.880              0.926    0.046
baseline net (MW/MW)                    0.003              0.007    0.004
events                                421.000             97.000 -324.000
median event duration (h)               3.000              3.500    0.500
median SoC at onset                     0.620              0.475   -0.145
preparedness gap                        0.350              0.519    0.169
dispatch gap                            0.520              0.317   -0.203
duration gap                            0.090              0.072   -0.018
response first half                     0.057              0.119    0.062
response second half                    0.051              0.103    0.052
above 50% of nameplate                  0.110              0.110   -0.000


## 8. What the modern era says

**The fleet responds about twice as hard.** Under its own top-percentile LoLP periods the
modern fleet nets **+0.115 MW per MW online** against the full window's +0.060, and discharges
in **93%** of them against 88%. Against a baseline that also roughly doubled (+0.007 vs
+0.003), so this is a stronger response, not just a busier fleet.

**And the binding constraint has moved.** This is the finding, and it reverses notebook 05's:

| gap | full window | modern era |
|---|---|---|
| preparedness — energy absent at onset | 35% | **52%** |
| dispatch — energy still held at the deepest point | **52%** | 32% |
| duration — time spent below the low-water mark | 9% | 7% |

Notebook 05 concludes the shortfall is **dispatch**: the fleet arrives reasonably full and then
holds half its energy back. Over the modern era that reverses. The dispatch gap falls from 52%
to 32% — the fleet is much readier to actually deploy what it holds — while the preparedness
gap climbs from 35% to 52%, and median state of charge at onset drops from 62% to **47%**.

Read together with notebook 07, that is a coherent story rather than a contradiction. Once the
control room could dispatch batteries, they began trading actively in ordinary conditions
rather than sitting idle — which is exactly what notebook 07 finds, with the largest change in
loose conditions rather than tight ones. A fleet that is working all day arrives at the next
tight period with less in the tank. The constraint moved from *"will anyone call on it"* to
*"is there anything left"*.

**The decay is now visible.** Response falls from +0.119 in an event's first half to +0.103 in
its second, a decay of −0.015 — three times the full window's −0.005 on a base twice the size.
A fleet that starts emptier and pushes harder runs down faster, and this is what that looks
like.

**Depth is unchanged.** 11.0% of site-periods above half of nameplate, identical to the full
window. The fleet responds more often and from a lower start, but the sites that push hard push
no harder than they used to.

**What this costs, and the two numbers not to quote.** The window holds a third of the periods
and two winters. `C_DRM<1GW` has **2 periods** and `C_CMN` has **3** — both are case studies and
neither supports a mean, which is why they are labelled rather than tabulated. The scarcity
thresholds also loosen sharply when recomputed within the era: LoLP's 99th percentile falls from
2.8e-04 to 7.9e-06, and the de-rated-margin floor rises from 3,235 MW to 5,260 MW. "The tightest
1% of the modern era" is a materially calmer set of periods than "the tightest 1% since 2018",
so part of the higher response is measured against an easier bar. Notebook 07's matched-margin
comparison is the control for that, and it holds.
